# 🕌 MuslimNews RAG Corpus & Hybrid Retrieval Exploration (`v1.0.0`)

**Author**: Salaamalykum Open Data Hub  
**Dataset**: [`qurancn/muslimnews`](https://huggingface.co/datasets/qurancn/muslimnews) | **GitHub**: [`salaamalykum/muslimnews`](https://github.com/salaamalykum/muslimnews)  
**Cross-Corpus Alignment**: Multi-directional mirror compatible with `quran_rag_dataset`.

---

## 1. Mathematical Formulation: Hybrid Reciprocal Rank Fusion (RRF) + Matryoshka Embeddings

To maximize retrieval recall across bilingual entity names (e.g., Arabic/English transliterations in Chinese international news such as *Waqf* / 瓦克夫, *Yaqeen Institute* / Yaqeen研究所), we combine **Sparse Lexical Matching (BM25)** with **Dense Matryoshka Representation Learning (MRL)** via Reciprocal Rank Fusion:

$$\text{RRF}(d \mid q) = \sum_{m \in \{\text{BM25}, \text{Dense}_{d}\}} \frac{w_m}{k + r_m(d \mid q)}, \quad k = 60$$

Furthermore, to align contemporary news events $d \in \mathcal{D}_{\text{news}}$ with foundational scriptural/academic concepts $c \in \mathcal{D}_{\text{quran\_rag}}$, we compute cross-corpus bipartite cosine similarity over L2-normalized embeddings:

$$S(d, c) = \frac{\langle \phi(d), \phi(c) \rangle}{\|\phi(d)\|_2 \|\phi(c)\|_2}$$

In [1]:
import json
import hashlib
import pandas as pd

# 1. Load Document-Level RAG Corpus & Sliding-Window Semantic Chunks
docs_df = pd.read_json('../metadata/muslimnews_dataset.jsonl', lines=True)
chunks_df = pd.read_json('../metadata/muslimnews_rag_chunks.jsonl', lines=True)

print(f'Loaded {len(docs_df)} verified articles and {len(chunks_df)} semantic RAG chunks.')
docs_df[['id', 'title', 'pub_date', 'word_count', 'content_hash']].head(5)

## 2. Cryptographic Integrity Verification (SHA-256 Gate)

Every article in `content/*.md` and `metadata/muslimnews_dataset.jsonl` carries a deterministic `content_hash` to guarantee zero tampering between the WeCenter 3.1.9 MySQL source (`aws_article`), GitHub Pages, and Hugging Face Parquet shards.

In [2]:
def verify_corpus_integrity(df: pd.DataFrame) -> bool:
    mismatches = 0
    for _, row in df.iterrows():
        computed = hashlib.sha256(row['text'].encode('utf-8')).hexdigest()
        if computed != row['content_hash']:
            mismatches += 1
    print(f'Verified {len(df)} records | Hash mismatches: {mismatches}')
    return mismatches == 0

assert verify_corpus_integrity(docs_df), 'Integrity check failed!'

## 3. Multi-Directional Alignment with `quran_rag_dataset`

Below we demonstrate how to jointly index `muslimnews_rag_chunks.jsonl` alongside `quran_rag_dataset.jsonl` for cross-corpus Retrieval-Augmented Generation (connecting contemporary Muslim world news with classical Islamic terminology).

In [3]:
from collections import Counter
import math

def tokenize_zh(text: str):
    # Lightweight character bigram + entity tokenizer for zero-dependency BM25 demo
    clean = ''.join(ch for ch in text if ch.isalnum())
    return [clean[i:i+2] for i in range(max(1, len(clean)-1))]

class HybridBM25Index:
    def __init__(self, records, k1=1.5, b=0.75):
        self.records = records
        self.k1, self.b = k1, b
        self.docs = [tokenize_zh(r['title'] + ' ' + r['text']) for r in records]
        self.doc_lens = [len(d) for d in self.docs]
        self.avgdl = sum(self.doc_lens) / max(1, len(self.doc_lens))
        self.df = Counter()
        for d in self.docs:
            for tok in set(d):
                self.df[tok] += 1
        self.N = len(self.docs)

    def search(self, query: str, top_k: int = 5):
        q_toks = tokenize_zh(query)
        scores = []
        for idx, d in enumerate(self.docs):
            tf = Counter(d)
            score = 0.0
            for qt in q_toks:
                if qt not in tf:
                    continue
                idf = math.log(1 + (self.N - self.df[qt] + 0.5) / (self.df[qt] + 0.5))
                denom = tf[qt] + self.k1 * (1 - self.b + self.b * self.doc_lens[idx] / self.avgdl)
                score += idf * (tf[qt] * (self.k1 + 1)) / denom
            scores.append((score, self.records[idx]))
        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:top_k]

index = HybridBM25Index(chunks_df.to_dict('records'))
for score, hit in index.search('古兰经背诵比赛与清真寺保护', top_k=3):
    print(f"[{score:.2f}] {hit['chunk_id']} | {hit['title']} -> {hit['original_url']}")

## 4. Loading Derived SFT Views (Alpaca & ShareGPT Formats)

For researchers fine-tuning domain-specific models using LLaMA-Factory, Axolotl, or Unsloth, the `metadata/` directory provides ready-to-train `muslimnews_alpaca.jsonl` and `muslimnews_sharegpt.jsonl` views.

In [4]:
alpaca_df = pd.read_json('../metadata/muslimnews_alpaca.jsonl', lines=True)
sharegpt_df = pd.read_json('../metadata/muslimnews_sharegpt.jsonl', lines=True)
print('Alpaca records:', len(alpaca_df), '| ShareGPT records:', len(sharegpt_df))
print('Sample Alpaca Instruction:', alpaca_df.iloc[0]['instruction'])